In [43]:
import pandas as pd
import numpy as np

import glob
import os
import os.path
import re
import sys
import warnings

from scipy.spatial import KDTree
from scipy.spatial import distance

from numpy.linalg import norm

#-------------------------------
from io import StringIO


import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap, ScalarMappable
from matplotlib.patches import Rectangle
import matplotlib.colors as colors

#from mpl_toolkits.mplot3d import Axes3D

In [44]:
#def main():

# ignore warnings
warnings.filterwarnings("ignore")

In [45]:
bond_mean = df_bonds['angl'].mean()
bond_disp = df_bonds['angl'].std(ddof=1)

print(filename)
print(bond_mean, bond_disp)

cascades_form_spectra_for_movie_npt/results_bonds_data_vs_z_h2_shrinking/bonds_angl_data_h2_layer38.0_shrinking_final_pyro_cascades_simulation0.xyz
119.44249976441264 1.6197071598104356


In [46]:
def compute_angle_between_3d_points(a,b,c):
    ba = a - b
    bc = c - b

    cosine_numerator = np.sum(ba*bc, axis=1)
    cosine_denominator_1 = np.linalg.norm(ba, axis=1)
    cosine_denominator_2 = np.linalg.norm(bc, axis=1)
    cosine_angle = cosine_numerator / (cosine_denominator_1 * cosine_denominator_2)
    angles = np.arccos(cosine_angle)
    degree_angles = np.rad2deg(angles)

    return degree_angles

In [47]:
def layer_category_fix_n(zmin,zmax,n):
    layers_list = []
    beginning = zmin
    h_step = (zmax-zmin)/n
    n_layer = 0
    while (beginning < zmax):
        layers_list.append([beginning, beginning+h_step, h_step, n_layer])
        beginning = beginning + h_step 
        n_layer += 1
    return sorted(layers_list)

In [50]:
def bond_angle_calc_mean():
    
    # ignore warnings
    warnings.filterwarnings("ignore")
    
    #layer_height = int(sys.argv[-1])
    layer_height = 2
            
    #Creatte directory to put avg results
    directory_frames = f'cascades_form_spectra_for_movie_npt/results_bonds_data_vs_z_h{layer_height}_shrinking'
    directory = f'{directory_frames}/results_bonds_vs_z_h{layer_height}_shrinking'
    try:
        os.makedirs(directory)
    except FileExistsError:
        # directory already exists
        pass
    

    # list of all files
    list_files = glob.glob(f'{directory_frames}/bonds_angl_data_h2_layer*_shrinking_final_pyro_cascades_simulation*.xyz')
    #list_files = [list_files[0]]
    
    list_sim = []
    for filename in list_files:
        list_sim.append(re.search('simulation(.+?).xyz', filename).group(1))
        
    list_sim = list(set(list_sim))
    #list_sim = [list_sim[0]]
    print(list_sim)
        
    
    for simulation in list_sim:
        print(simulation)
        # load data from file
        filename = f"cascades_form_spectra_for_movie_npt/final_pyro_cascades_simulation{simulation}.xyz"
        input_data = open(filename).read()
        df_atoms = pd.read_table(filename, skiprows=9,
                                       names=['element', 'x', 'y', 'z', 'id', 'coord_num', 'pot_eng'],
                                       sep='\\s+')
        df_atoms = df_atoms.drop(['element'],axis=1)
            
                       
        # assign z_layer_fix_n
        init_intf = 6
        init_z_min = -169.64307163528616
        init_z_max = 150.98293179686004
        n = (init_z_max - init_z_min) //layer_height
            
        layers = layer_category_fix_n(df_atoms['z'].min(),df_atoms['z'].max(), n)        
        #print(layers)
        array=np.array(layers)        
        df_final = pd.DataFrame(array,columns=['beginning','ending','height','z_layers'])
        df_final['mean_bond_angl'] = 0
        df_final['disp_bond_angl'] = 0
        
         # list of files for certain simulation
        list_files = glob.glob(f'{directory_frames}/bonds_angl_data_h2_layer*_shrinking_final_pyro_cascades_simulation{simulation}.xyz')
        count_files = 1
        for filename in list_files:
            n = 73
            print(count_files, "/", len(list_files), "file name:", filename[n:])            
            layer = re.search('layer(.+?)_shrinking', filename).group(1)
        
            #load bond length data
            df_bonds = pd.read_table(filename, header=0, sep=',', index_col=0)
            
            a = df_bonds[['x_neigb1_atom','y_neigb1_atom','z_neigb1_atom']].to_numpy()
            b = df_bonds[['x_central_atom','y_central_atom','z_central_atom']].to_numpy()
            c = df_bonds[['x_neigb2_atom','y_neigb2_atom','z_neigb2_atom']].to_numpy()
            df_bonds['angl'] = compute_angle_between_3d_points(a,b,c)
            
            bond_mean = df_bonds['angl'].mean()
            bond_disp = df_bonds['angl'].std(ddof=1)
            
            df_final.loc[df_final['z_layers'] == float(layer), 'mean_bond_angl'] = bond_mean
            df_final.loc[df_final['z_layers'] == float(layer), 'disp_bond_angl'] = bond_disp
            print (layer, bond_mean, bond_disp)
            display(df_final[df_final['z_layers'] == float(layer)])
        
            count_files += 1
        
        border_factor = 0
        if border_factor == 0:
            file_angl = f'{directory}/bonds_angl_h{layer_height}_shrinking_final_pyro_cascades_simulation{simulation}.xyz'
        else:
            file_angl = f'{directory}/bonds_angl_h{layer_height}_b{border_factor}_shrinking_final_pyro_cascades_simulation{simulation}.xyz'
        
        
        df_final.to_csv(file_angl, sep=',')
        print("Done")

In [51]:
bond_angle_calc_mean()

['200', '350', '0', '500']
200
1 / 160 file name: bonds_angl_data_h2_layer115.0_shrinking_final_pyro_cascades_simulation200.xyz
115.0 115.78729756955934 8.3259526817057


,beginning,ending,height,z_layers,mean_bond_angl,disp_bond_angl
115,80.74425,83.3178,2.57355,115.0,115.787298,8.325953


2 / 160 file name: bonds_angl_data_h2_layer148.0_shrinking_final_pyro_cascades_simulation200.xyz
148.0 115.60784691753257 8.577974691757888


,beginning,ending,height,z_layers,mean_bond_angl,disp_bond_angl
148,165.6714,168.24495,2.57355,148.0,115.607847,8.577975


3 / 160 file name: bonds_angl_data_h2_layer133.0_shrinking_final_pyro_cascades_simulation200.xyz
133.0 116.01278688799648 8.002062894587633


...

159 / 160 file name: bonds_angl_data_h2_layer155.0_shrinking_final_pyro_cascades_simulation500.xyz
155.0 116.48749716738394 7.8311377152361725


,beginning,ending,height,z_layers,mean_bond_angl,disp_bond_angl
155,185.701844,188.302275,2.600431,155.0,116.487497,7.831138


160 / 160 file name: bonds_angl_data_h2_layer2.0_shrinking_final_pyro_cascades_simulation500.xyz
2.0 116.73352586677156 7.4175100073113


,beginning,ending,height,z_layers,mean_bond_angl,disp_bond_angl
2,-212.164137,-209.563706,2.600431,2.0,116.733526,7.41751


Done
